# 01 — Object Generation

In [1]:
import scanpy as sc
import pandas as pd
import os

sc.logging.print_header()

Package,Version
scanpy,1.11.5
pandas,2.3.3
Component,Info
Python,"3.11.14 | packaged by conda-forge | (main, Jan 27 2026, 00:00:54) [Clang 19.1.7 ]"
OS,macOS-15.7.4-x86_64-i386-64bit
CPU,"6 logical CPU cores, i386"
GPU,No GPU found
Updated,2026-03-24 17:07
Dependency,Version
debugpy,1.8.20


In [2]:
raw_dir = '../data/raw'
processed_dir = '../data/processed'

run_ids = [
    'output-XETG00171__0043340__29059__20240913__193721',
    'output-XETG00171__0043340__30373__20240913__193721',
    'output-XETG00171__0043352__29072__20240913__193721',
    'output-XETG00171__0043352__29964__20240913__193721',
]

file_paths   = [os.path.join(raw_dir, r, 'cell_feature_matrix.h5') for r in run_ids]
spatial_paths = [os.path.join(raw_dir, r, 'cells.csv')             for r in run_ids]

In [3]:
adatas = [sc.read_10x_h5(f) for f in file_paths]

In [4]:
for i, spatial_csv in enumerate(spatial_paths):
    df_spatial = pd.read_csv(spatial_csv)
    df_spatial.set_index(adatas[i].obs_names, inplace=True)
    adatas[i].obs = df_spatial.copy()
    adatas[i].obsm['spatial'] = adatas[i].obs[['x_centroid', 'y_centroid']].to_numpy()

adata = adatas[0].concatenate(*adatas[1:])

/var/folders/h2/fwd571gj1xnggpdwy9194m8r0000gn/T/ipykernel_72584/469741856.py:9: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = adatas[0].concatenate(*adatas[1:])


In [5]:
batch_to_region = {
    '0': '29059',  # Kras-H3K36M-Setd2KO
    '1': '30373',  # Kras-Ctrl
    '2': '29072',  # Kras-Setd2KO
    '3': '29964',  # Kras-H3K36M
}
genotype_mapping = {
    '30373': 'Kras-Ctrl',
    '29072': 'Kras-Setd2KO',
    '29964': 'Kras-H3K36M',
    '29059': 'Kras-H3K36M-Setd2KO',
}
adata.obs['Genotype'] = adata.obs['batch'].map(batch_to_region).map(genotype_mapping)

print(adata.obs['Genotype'].value_counts())

Genotype
Kras-H3K36M-Setd2KO    485208
Kras-Ctrl              371513
Kras-H3K36M            306612
Kras-Setd2KO           179948
Name: count, dtype: int64


In [6]:
adata.write_h5ad(os.path.join(processed_dir, 'Gladstein object.h5ad'))